# Deep Learning Model for Deepfake Voice Detection — v2 (Improved Spoof Recall)

**All five fixes applied:**

| # | Fix | Expected gain |
|---|-----|---------------|
| 1 | Threshold search & checkpointing on **spoof_recall** | Directly aligns training with metric |
| 2 | **FocalLoss** replaces CrossEntropyLoss | Biggest single improvement for imbalance |
| 3 | **WeightedRandomSampler** — balanced batches | Spoof gradient every step |
| 4 | **AttentiveStatisticsPooling** replaces AvgPool | Preserves temporal artefacts |
| 5 | **SpecAugment** during training | Better generalisation |

## 1. Install dependencies

Run once per Colab session.

In [4]:
!pip -q install torch torchaudio scikit-learn pandas matplotlib tqdm optuna
print("Dependencies installed.")

Dependencies installed.


## 2. Imports

In [5]:
# ─── 2. Imports ───────────────────────────────────────────────────────────────
import json
import math
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_curve,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm

## 3. Mount Google Drive & set paths

In [6]:
# ─── 3. Mount Google Drive (Colab only) ───────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/DL Project")
except Exception:
    PROJECT_ROOT = Path("DL Project")

TRAIN_AUDIO_DIR       = PROJECT_ROOT / "flac_T"
TEST_AUDIO_DIR        = PROJECT_ROOT / "flac_D"
TRAIN_META_PATH       = PROJECT_ROOT / "ASVspoof5.train.metadata.txt"
TEST_META_PATH        = PROJECT_ROOT / "ASVspoof5.dev.metadata.txt"

OUTPUT_DIR            = PROJECT_ROOT / "outputs_asvspoof"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CLEAN_META_PATH     = OUTPUT_DIR / "train_metadata_filtered.txt"
DEV_CLEAN_META_PATH       = OUTPUT_DIR / "dev_metadata_filtered.txt"
TRAIN_SOURCE_REMOVED_PATH = OUTPUT_DIR / "train_missing_entries.csv"
DEV_SOURCE_REMOVED_PATH   = OUTPUT_DIR / "dev_missing_entries.csv"
TRAIN_95_5_META_PATH      = OUTPUT_DIR / "train_95_5.txt"
VAL_95_5_META_PATH        = OUTPUT_DIR / "val_95_5.txt"
TEST_95_5_META_PATH       = OUTPUT_DIR / "test_95_5.txt"
TRAIN_CLEANED_PATH        = OUTPUT_DIR / "train_95_5_cleaned.csv"
VAL_CLEANED_PATH          = OUTPUT_DIR / "val_95_5_cleaned.csv"
TEST_CLEANED_PATH         = OUTPUT_DIR / "test_95_5_cleaned.csv"

SAVE_DIR = PROJECT_ROOT / "saved_models_v7"
SAVE_DIR.mkdir(exist_ok=True)

for p, label in [
    (PROJECT_ROOT,    "PROJECT_ROOT"),
    (TRAIN_AUDIO_DIR, "TRAIN_AUDIO_DIR"),
    (TEST_AUDIO_DIR,  "TEST_AUDIO_DIR"),
    (TRAIN_META_PATH, "TRAIN_META_PATH"),
    (TEST_META_PATH,  "TEST_META_PATH"),
]:
    status = "OK" if p.exists() else "MISSING"
    print(f"{label}: {status} -> {p}")
    if status == "MISSING":
        raise FileNotFoundError(f"Required path not found: {p}")

print("\nPaths OK.")

Mounted at /content/drive
PROJECT_ROOT: OK -> /content/drive/MyDrive/DL Project
TRAIN_AUDIO_DIR: OK -> /content/drive/MyDrive/DL Project/flac_T
TEST_AUDIO_DIR: OK -> /content/drive/MyDrive/DL Project/flac_D
TRAIN_META_PATH: OK -> /content/drive/MyDrive/DL Project/ASVspoof5.train.metadata.txt
TEST_META_PATH: OK -> /content/drive/MyDrive/DL Project/ASVspoof5.dev.metadata.txt

Paths OK.


## 4. Global config & reproducibility

In [7]:
# ─── 4. Global config ─────────────────────────────────────────────────────────
SEED                 = 42
DEVICE               = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE          = 16000
MAX_SECONDS          = 4.0
MAX_SAMPLES          = int(SAMPLE_RATE * MAX_SECONDS)
N_MELS               = 128
FFT_SIZE             = 1024
HOP_LENGTH           = 160
BATCH_SIZE           = 32
NUM_WORKERS          = 0
TARGET_BONAFIDE_RATIO = 0.95

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"DEVICE: {DEVICE}")

DEVICE: cuda


## 5. Metadata helpers

In [8]:
# ─── 5. Metadata helpers ──────────────────────────────────────────────────────
def load_raw_metadata(meta_path):
    return pd.read_csv(meta_path, sep=r"\s+", header=None, engine="python")


def standardize_raw_split_df(df, audio_dir, file_col, file_mode, label_col):
    out = pd.DataFrame()
    out["file_id"]    = df[file_col].astype(str).str.strip()
    out["label_text"] = df[label_col].astype(str).str.lower().str.strip()
    out["label"]      = out["label_text"].map({"bonafide": 0, "spoof": 1})
    out["audio_path"] = out["file_id"].apply(
        lambda fid: str(audio_dir / f"{fid}.flac")
    )
    n_before = len(out)
    out = out.dropna(subset=["label"]).reset_index(drop=True)
    out["label"] = out["label"].astype(int)
    if len(out) < n_before:
        print(f"[standardize] Dropped {n_before - len(out)} rows with unrecognised labels.")
    return out


def clean_standardized_subset(df, name):
    df          = df.copy()
    exists_mask = df["audio_path"].apply(os.path.exists)
    kept_df     = df[exists_mask].reset_index(drop=True)
    removed_df  = df[~exists_mask].reset_index(drop=True)
    print(f"[{name}] total={len(df)} | kept={len(kept_df)} | removed={len(removed_df)}")
    return kept_df, removed_df


def sample_to_ratio(df, bonafide_ratio=0.95, seed=SEED):
    df       = df.copy()
    bona_df  = df[df["label"] == 0]
    spoof_df = df[df["label"] == 1]
    n_bona, n_spoof = len(bona_df), len(spoof_df)
    if n_bona == 0 or n_spoof == 0:
        raise ValueError("Both bonafide and spoof samples are required.")
    max_spoof = math.floor(n_bona * (1 - bonafide_ratio) / bonafide_ratio)
    if max_spoof <= n_spoof:
        sampled_bona  = bona_df
        sampled_spoof = spoof_df.sample(n=max_spoof, random_state=seed)
    else:
        target_bona  = math.floor(n_spoof * bonafide_ratio / (1 - bonafide_ratio))
        sampled_bona  = bona_df.sample(n=target_bona, random_state=seed)
        sampled_spoof = spoof_df
    out = pd.concat([sampled_bona, sampled_spoof], axis=0)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def save_standardized_metadata(df, path):
    df[["file_id", "label_text", "label", "audio_path"]].to_csv(path, index=False)
    print(f"Saved metadata to: {path}")


print("Metadata helpers defined.")

Metadata helpers defined.


## 6. Build / reload train · val · test splits

If cleaned CSVs already exist in your Drive, the build step is skipped automatically.

In [9]:
# ─── 6. Build / reload train/val/test splits ──────────────────────────────────
# If cleaned CSVs already exist in Drive, skip straight to the reload block.

if not TRAIN_CLEANED_PATH.exists():
    print("Building splits from raw metadata ...")
    TRAIN_FILE_COL  = 1
    TRAIN_LABEL_COL = 5
    TEST_FILE_COL   = 1
    TEST_LABEL_COL  = 5

    train_meta_raw = load_raw_metadata(TRAIN_META_PATH)
    test_meta_raw  = load_raw_metadata(TEST_META_PATH)

    train_source_raw = standardize_raw_split_df(
        train_meta_raw, TRAIN_AUDIO_DIR, TRAIN_FILE_COL, "T", TRAIN_LABEL_COL
    )
    dev_source_raw = standardize_raw_split_df(
        test_meta_raw, TEST_AUDIO_DIR, TEST_FILE_COL, "D", TEST_LABEL_COL
    )

    train_source_df, _ = clean_standardized_subset(train_source_raw, "train_source_clean")
    dev_source_df,   _ = clean_standardized_subset(dev_source_raw,   "dev_source_clean")

    save_standardized_metadata(train_source_df, TRAIN_CLEAN_META_PATH)
    save_standardized_metadata(dev_source_df,   DEV_CLEAN_META_PATH)

    train_df = sample_to_ratio(train_source_df, bonafide_ratio=TARGET_BONAFIDE_RATIO)
    dev_95_5 = sample_to_ratio(dev_source_df,   bonafide_ratio=TARGET_BONAFIDE_RATIO)

    val_df, test_df = train_test_split(
        dev_95_5, test_size=0.50, random_state=SEED, stratify=dev_95_5["label"]
    )
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    train_df.to_csv(TRAIN_CLEANED_PATH, index=False)
    val_df.to_csv(VAL_CLEANED_PATH,     index=False)
    test_df.to_csv(TEST_CLEANED_PATH,   index=False)
    print("Splits saved.")
else:
    print("Cleaned CSVs found — skipping rebuild.")

# Always reload from CSV to guarantee consistent dtypes
train_df = pd.read_csv(TRAIN_CLEANED_PATH)
val_df   = pd.read_csv(VAL_CLEANED_PATH)
test_df  = pd.read_csv(TEST_CLEANED_PATH)

for name, df in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
    n, s, b = len(df), (df["label"] == 1).sum(), (df["label"] == 0).sum()
    print(f"  {name}: {n} total | bonafide={b} ({b/n:.1%}) | spoof={s} ({s/n:.1%})")

print("Datasets loaded.")

Cleaned CSVs found — skipping rebuild.
  TRAIN: 19786 total | bonafide=18797 (95.0%) | spoof=989 (5.0%)
  VAL: 15991 total | bonafide=15191 (95.0%) | spoof=800 (5.0%)
  TEST: 15991 total | bonafide=15192 (95.0%) | spoof=799 (5.0%)
Datasets loaded.


## 7. Feature extraction + SpecAugment

**Fix 5** — `SpecAugment` (FrequencyMasking + TimeMasking) is applied **only during training** to improve generalisation.

In [10]:
# ─── 7. Feature extraction + SpecAugment (Fix 5) ─────────────────────────────
class LogMelFeatureExtractor(nn.Module):
    def __init__(self, sample_rate=16_000, n_mels=128, n_fft=1024, hop_length=160):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            power=2.0,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power")

    def forward(self, waveform):
        return self.db(self.mel(waveform))


# FIX 5: SpecAugment applied ONLY during training
train_spec_augment = nn.Sequential(
    T.FrequencyMasking(freq_mask_param=20),   # mask up to 20 mel bins
    T.TimeMasking(time_mask_param=40),         # mask up to 40 time frames
)


def load_and_fix_length(audio_path, target_sr=SAMPLE_RATE, max_samples=MAX_SAMPLES):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    n = waveform.shape[-1]
    if n < max_samples:
        waveform = F.pad(waveform, (0, max_samples - n))
    else:
        waveform = waveform[:, :max_samples]
    return waveform


class SpoofDataset(Dataset):
    def __init__(self, df, feature_extractor, augment=None):
        self.df                = df.reset_index(drop=True)
        self.feature_extractor = feature_extractor
        self.augment           = augment   # None for val / test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        waveform = load_and_fix_length(row["audio_path"])
        features = self.feature_extractor(waveform)          # [1, n_mels, time]
        if self.augment is not None:
            features = self.augment(features)
        label = torch.tensor(row["label"], dtype=torch.long)
        return features, label, row["file_id"]


feature_extractor = LogMelFeatureExtractor(
    sample_rate=SAMPLE_RATE, n_mels=N_MELS, n_fft=FFT_SIZE, hop_length=HOP_LENGTH
)

## 8. DataLoaders with `WeightedRandomSampler`

**Fix 3** — each training mini-batch is sampled to be ~50 % spoof / 50 % bonafide, regardless of the 95/5 underlying split. The batch label distribution is printed as a sanity check.

In [11]:
# ─── 8. DataLoaders with WeightedRandomSampler (Fix 3) ───────────────────────
# Each training batch will be roughly 50% spoof, 50% bonafide regardless of
# the 95/5 split in the underlying dataset.

def make_weighted_sampler(labels):
    counts           = Counter(labels.tolist())
    weight_per_class = {cls: 1.0 / count for cls, count in counts.items()}
    sample_weights   = [weight_per_class[l] for l in labels.tolist()]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


train_dataset = SpoofDataset(train_df, feature_extractor, augment=train_spec_augment)
val_dataset   = SpoofDataset(val_df,   feature_extractor, augment=None)
test_dataset  = SpoofDataset(test_df,  feature_extractor, augment=None)

train_sampler = make_weighted_sampler(train_df["label"])

_loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
train_loader   = DataLoader(train_dataset, sampler=train_sampler, **_loader_kwargs)
val_loader     = DataLoader(val_dataset,   shuffle=False,          **_loader_kwargs)
test_loader    = DataLoader(test_dataset,  shuffle=False,          **_loader_kwargs)

features, labels, _ = next(iter(train_loader))
print(f"Feature batch shape : {features.shape}")
print(f"Batch label balance : {Counter(labels.tolist())}  (~50/50)")

Feature batch shape : torch.Size([32, 1, 128, 401])
Batch label balance : Counter({1: 19, 0: 13})  (~50/50)


## 9. Model architecture — `AttentiveStatisticsPooling`

**Fix 4** — `AdaptiveAvgPool2d(1,1)` is replaced with `AttentiveStatisticsPooling`. Instead of averaging all time frames, the model learns which frames are most discriminative for spoof detection. Output size goes from `c3` → `2·c3` (mean + std), so the first linear layer input doubles to 256.

In [12]:
# ─── 9. Model Architecture with AttentiveStatisticsPooling (Fix 4) ────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.block(x)


class AttentiveStatisticsPooling(nn.Module):
    """
    Replaces AdaptiveAvgPool2d(1,1).

    Input:  [B, C, H, T]  (CNN feature maps)
    Output: [B, 2*C]      (weighted mean + std concatenated)

    Instead of averaging all time frames uniformly, this module learns a
    per-channel attention score over time, letting the model emphasise
    the frames that are most discriminative for spoof detection (e.g.
    vocoder artefacts, unnatural pitch transitions).
    """
    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Conv1d(in_channels, in_channels, kernel_size=1)

    def forward(self, x):
        # x: [B, C, H, T]
        x       = x.mean(dim=2)                               # [B, C, T]  collapse freq
        weights = torch.softmax(self.attn(x), dim=-1)         # [B, C, T]  attention
        mean    = (weights * x).sum(dim=-1)                   # [B, C]
        var     = (weights * (x - mean.unsqueeze(-1)) ** 2).sum(dim=-1)
        std     = (var + 1e-8).sqrt()                         # [B, C]
        return torch.cat([mean, std], dim=1)                  # [B, 2*C]


class SpoofCNN(nn.Module):
    def __init__(self, base_channels=32, dropout=0.3, num_classes=2):
        super().__init__()
        c1, c2, c3 = base_channels, base_channels * 2, base_channels * 4

        self.features = nn.Sequential(
            ConvBlock(1,  c1, dropout=dropout / 2),
            ConvBlock(c1, c2, dropout=dropout / 2),
            ConvBlock(c2, c3, dropout=dropout),
        )
        # FIX 4: attentive pooling — output size is 2*c3 (mean + std)
        self.pool = AttentiveStatisticsPooling(c3)

        self.classifier = nn.Sequential(
            nn.Linear(c3 * 2, c2),   # 2*c3 input because of mean+std concat
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(c2, num_classes),
        )

    def forward(self, x):
        feat   = self.features(x)     # [B, c3, H', T']
        pooled = self.pool(feat)      # [B, 2*c3]
        return self.classifier(pooled)


# Smoke-test
_test_model = SpoofCNN(base_channels=32, dropout=0.3).to(DEVICE)
with torch.no_grad():
    _dummy = torch.zeros(2, 1, N_MELS, 401).to(DEVICE)
    _out   = _test_model(_dummy)
    assert _out.shape == (2, 2), f"Unexpected output shape: {_out.shape}"
print(_test_model)
total_params = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")
del _test_model, _dummy, _out

SpoofCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Dropout(p=0.15, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1

## 10. `FocalLoss`

**Fix 2** — `FocalLoss` replaces `CrossEntropyLoss`. `alpha` (spoof class weight) and `gamma` (focusing exponent) are both tuned by Optuna, so the search finds the right imbalance trade-off automatically.

In [13]:
# ─── 10. FocalLoss (Fix 2) ────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    Focal Loss for severe class imbalance.

    alpha : weight for the positive (spoof) class.
            Recommended range 0.60-0.90 for 95/5 imbalance.
            Higher = more weight on spoof recall, lower = fewer false alarms.
    gamma : focusing exponent.  Higher values concentrate gradient on
            hard / misclassified examples (typically 2.0).

    How it helps vs CrossEntropyLoss:
      - CrossEntropy with class_weights adjusts the loss scale but every
        easy well-classified bonafide sample still contributes gradient.
      - FocalLoss multiplies each sample's loss by (1-p_t)^gamma, which
        near-zeros the contribution from confidently-correct bonafide
        samples, so the gradient is dominated by the rare spoof examples.
    """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction="none")
        pt   = torch.exp(-ce)
        # alpha for spoof (1), (1-alpha) for bonafide (0)
        a_t  = torch.where(
            targets == 1,
            torch.full_like(ce, self.alpha),
            torch.full_like(ce, 1.0 - self.alpha),
        )
        loss = a_t * (1.0 - pt) ** self.gamma * ce
        return loss.mean()

## 11. Metrics & threshold search

**Fix 1** — `find_best_threshold` now maximises **spoof recall** subject to `spoof_precision ≥ 0.40`, rather than maximising generic binary F1. Model checkpointing uses `spoof_recall` as the criterion.

In [14]:
# ─── 11. Metrics and threshold search (Fix 1) ─────────────────────────────────

def compute_eer(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fnr[idx] + fpr[idx]) / 2.0), float(thresholds[idx])


def compute_spoof_metrics(y_true, y_prob, threshold=0.5):
    """Full per-class breakdown: spoof recall, precision, F1, EER, confusion."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    m = {}
    m["threshold"]         = float(threshold)
    m["accuracy"]          = accuracy_score(y_true, y_pred)
    m["balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)
    m["spoof_precision"]   = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    m["spoof_recall"]      = recall_score(y_true,   y_pred,  pos_label=1, zero_division=0)
    m["spoof_f1"]          = f1_score(y_true, y_pred,        pos_label=1, zero_division=0)
    m["bonafide_recall"]   = recall_score(y_true, y_pred,    pos_label=0, zero_division=0)
    m["macro_f1"]          = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    m["weighted_f1"]       = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    try:
        m["roc_auc"]       = roc_auc_score(y_true, y_prob)
    except Exception:
        m["roc_auc"]       = float("nan")
    try:
        m["eer"], m["eer_threshold"] = compute_eer(y_true, y_prob)
    except Exception:
        m["eer"] = m["eer_threshold"] = float("nan")

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    m["tn"], m["fp"], m["fn"], m["tp"] = int(tn), int(fp), int(fn), int(tp)
    m["false_positive_rate"] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    m["false_negative_rate"] = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return m, y_pred


def find_best_threshold(y_true, y_prob, min_spoof_precision=0.40):
    """
    FIX 1 — Threshold search that maximises spoof recall while keeping
    spoof precision >= min_spoof_precision.
    """
    best_thr_recall, best_recall = 0.5, -1.0
    best_thr_f1,     best_f1    = 0.5, -1.0

    for thr_int in range(2, 71):          # 0.02 to 0.70 in steps of 0.01
        thr    = thr_int / 100.0
        y_pred = (np.asarray(y_prob) >= thr).astype(int)
        s_prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        s_rec  = recall_score(y_true,   y_pred,  pos_label=1, zero_division=0)
        mf1    = f1_score(y_true, y_pred, average="macro",    zero_division=0)

        if mf1 > best_f1:
            best_f1    = mf1
            best_thr_f1 = thr

        if s_prec >= min_spoof_precision and s_rec > best_recall:
            best_recall    = s_rec
            best_thr_recall = thr

    best_thr = best_thr_recall if best_recall > 0 else best_thr_f1
    metrics, _ = compute_spoof_metrics(y_true, y_prob, threshold=best_thr)
    return best_thr, metrics


print("Loss, metrics, and threshold search defined.")

Loss, metrics, and threshold search defined.


## 12. Training & evaluation functions

In [19]:
# ─── 12. Training and evaluation ──────────────────────────────────────────────

def evaluate_model(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss, all_probs, all_labels, all_file_ids = 0.0, [], [], []
    with torch.no_grad():
        for x, y, file_ids in loader:
            x, y    = x.to(DEVICE), y.to(DEVICE)
            logits  = model(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            probs   = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
            all_file_ids.extend(list(file_ids))
    avg_loss = total_loss / len(loader.dataset)
    metrics, _ = compute_spoof_metrics(all_labels, all_probs, threshold=threshold)
    metrics["loss"] = float(avg_loss)
    return metrics, np.array(all_probs), np.array(all_labels), np.array(all_file_ids)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for x, y, _ in tqdm(loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)


def fit_model(
    model,
    train_loader,
    val_loader,
    epochs,
    lr,
    weight_decay,
    focal_alpha=0.75,
    focal_gamma=2.0,
    patience=7,
    trial=None,
):
    # FIX 2: Focal loss — no class_weights needed (alpha handles it)
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    # FIX 1: scheduler monitors macro_f1 (balanced across both classes)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    history = {k: [] for k in [
        "train_loss", "val_loss", "val_accuracy",
        "val_spoof_recall", "val_macro_f1", "val_roc_auc", "val_best_threshold",
    ]}
    best_state, best_spoof_recall = None, -1.0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)

        # Evaluate at 0.5 just to get the loss value
        val_metrics_05, val_probs, val_labels, _ = evaluate_model(
            model, val_loader, criterion, threshold=0.5
        )
        # FIX 1: find threshold that maximises spoof recall
        best_thr, tuned_metrics = find_best_threshold(val_labels, val_probs)
        tuned_metrics["loss"]   = val_metrics_05["loss"]

        # FIX 1: scheduler steps on macro_f1
        scheduler.step(tuned_metrics["macro_f1"])

        for key, val in [
            ("train_loss",         train_loss),
            ("val_loss",           tuned_metrics["loss"]),
            ("val_accuracy",       tuned_metrics["accuracy"]),
            ("val_spoof_recall",   tuned_metrics["spoof_recall"]),
            ("val_macro_f1",       tuned_metrics["macro_f1"]),
            ("val_roc_auc",        tuned_metrics["roc_auc"]),
            ("val_best_threshold", best_thr),
        ]:
            history[key].append(val)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={tuned_metrics['loss']:.4f} | "
            f"acc={tuned_metrics['accuracy']:.4f} | "
            f"spoof_rec={tuned_metrics['spoof_recall']:.4f} | "
            f"spoof_prec={tuned_metrics['spoof_precision']:.4f} | "
            f"macro_f1={tuned_metrics['macro_f1']:.4f} | "
            f"roc_auc={tuned_metrics['roc_auc']:.4f} | "
            f"thr={best_thr:.2f}"
        )

        # FIX 1: checkpoint on spoof_recall instead of generic F1
        if tuned_metrics["spoof_recall"] > best_spoof_recall:
            best_spoof_recall = tuned_metrics["spoof_recall"]
            best_state = {
                "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "best_val_metrics": tuned_metrics,
                "best_threshold":   float(best_thr),
                "epoch":            epoch,
            }
            save_dir = SAVE_DIR / f"lr{lr:.6f}_decay{weight_decay:.6f}"
            save_dir.mkdir(parents=True, exist_ok=True)
            ckpt_path = save_dir / f"epoch{epoch:02d}_spoof_rec{best_spoof_recall:.4f}.pt"
            torch.save(model.state_dict(), ckpt_path)
            print(f"  *** Improved (spoof_recall={best_spoof_recall:.4f}), saved -> {ckpt_path}")

        # Early stopping: patience on spoof_recall
        epochs_since_best = (
            len(history["val_spoof_recall"])
            - history["val_spoof_recall"].index(best_spoof_recall)
            - 1
        )
        if epochs_since_best >= patience:
            print(f"Early stopping: no spoof_recall improvement for {patience} epochs.")
            break

        # Optuna pruning
        if trial is not None:
            trial.report(tuned_metrics["macro_f1"], epoch)
            if trial.should_prune():
                import optuna as _optuna
                raise _optuna.TrialPruned()

    return history, best_state


print("Training functions defined.")

Training functions defined.


## 13. Optuna hyperparameter search

**Fix 1 + 2** — Optuna now searches over `focal_alpha` and `focal_gamma` in addition to `lr`, `weight_decay`, and `dropout`. The objective maximises `macro_f1` (penalises low spoof recall strongly). `SAVE_DIR = saved_models_v7` so existing checkpoints are untouched.

In [13]:
# ─── 13. Optuna hyperparameter search ─────────────────────────────────────────
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

global_best = {"spoof_recall": -1.0, "macro_f1": -1.0, "state": None, "params": None}


def objective(trial):
    # FIX 1+2: search space includes focal loss params
    lr           = trial.suggest_float("lr",           1e-5, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    dropout      = trial.suggest_float("dropout",      0.2,  0.5,  step=0.1)
    focal_alpha  = trial.suggest_float("focal_alpha",  0.60, 0.90, step=0.05)
    focal_gamma  = trial.suggest_float("focal_gamma",  1.0,  3.0,  step=0.5)

    m = SpoofCNN(base_channels=32, dropout=dropout).to(DEVICE)

    history, best_state = fit_model(
        m,
        train_loader,
        val_loader,
        epochs=10,
        lr=lr,
        weight_decay=weight_decay,
        focal_alpha=focal_alpha,
        focal_gamma=focal_gamma,
        patience=3,
        trial=trial,
    )

    if best_state is None:
        return 0.0

    trial_macro_f1    = best_state["best_val_metrics"]["macro_f1"]
    trial_spoof_recall = best_state["best_val_metrics"]["spoof_recall"]

    # Save globally best model
    if trial_macro_f1 > global_best["macro_f1"]:
        global_best["macro_f1"]     = trial_macro_f1
        global_best["spoof_recall"] = trial_spoof_recall
        global_best["state"]        = best_state
        global_best["params"]       = trial.params

        gb_dir = SAVE_DIR / "global_best"
        gb_dir.mkdir(parents=True, exist_ok=True)
        torch.save(
            {**best_state, "trial_params": trial.params},
            gb_dir / "global_best_model.pt",
        )
        print(
            f"  *** New global best: "
            f"macro_f1={trial_macro_f1:.4f} | "
            f"spoof_recall={trial_spoof_recall:.4f} | "
            f"params={trial.params}"
        )

    return trial_macro_f1   # FIX 1: Optuna maximises macro_f1


def run_optuna_search(n_trials=10):
    sampler = TPESampler(seed=SEED)
    pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=n_trials, n_jobs=1)

    print("\n===== Optuna Search Complete =====")
    print(f"Best trial macro_f1   : {study.best_value:.4f}")
    print(f"Best params           : {study.best_params}")
    print(f"Global best macro_f1  : {global_best['macro_f1']:.4f}")
    print(f"Global best spoof_rec : {global_best['spoof_recall']:.4f}")
    return study


study = run_optuna_search(n_trials=10)

[I 2026-04-17 18:01:10,424] A new study created in memory with name: no-name-79871e29-2531-4982-b9f6-9aefbd91407c


  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 01/10 | train_loss=0.0253 | val_loss=0.0329 | acc=0.9315 | spoof_rec=0.8013 | spoof_prec=0.4065 | macro_f1=0.7512 | roc_auc=0.8949 | thr=0.30
  *** Improved (spoof_recall=0.8013), saved -> /content/drive/MyDrive/DL Project/saved_models_v6/lr0.000315_decay0.007114/epoch01_spoof_rec0.8013.pt


  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 02/10 | train_loss=0.0118 | val_loss=0.0327 | acc=0.9345 | spoof_rec=0.7588 | spoof_prec=0.4155 | macro_f1=0.7509 | roc_auc=0.8822 | thr=0.31


...

  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 06/10 | train_loss=0.0021 | val_loss=0.8060 | acc=0.0530 | spoof_rec=1.0000 | spoof_prec=0.0502 | macro_f1=0.0509 | roc_auc=0.6797 | thr=0.70
  *** Improved (spoof_recall=1.0000), saved -> /content/drive/MyDrive/DL Project/saved_models_v6/lr0.000042_decay0.000015/epoch06_spoof_rec1.0000.pt


  0%|          | 0/619 [00:00<?, ?it/s]

[W 2026-04-17 23:51:17,660] Trial 1 failed with parameters: {'lr': 4.207053950287933e-05, 'weight_decay': 1.493656855461762e-05, 'dropout': 0.5, 'focal_alpha': 0.8, 'focal_gamma': 2.5} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_4976/2103006657.py", line 19, in objective
    history, best_state = fit_model(
                          ^^^^^^^^^^
  File "/tmp/ipykernel_4976/4189421893.py", line 66, in fit_model
    val_metrics_05, val_probs, val_labels, _ = evaluate_model(
                                               ^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4976/4189421893.py", line 7, in evaluate_model
    for x, y, file_ids in loader:
                          ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741,

KeyboardInterrupt: 

## 14. Final training with best hyperparameters

In [18]:
best_params = {
    "lr":           0.0003148911647956861,
    "weight_decay": 0.007114476009343225,
    "dropout":      0.4,
    "focal_alpha":  0.8,
    "focal_gamma":  1.0,
}

set_seed(SEED)
final_model = SpoofCNN(base_channels=32, dropout=best_params["dropout"]).to(DEVICE)

final_history, final_best_state = fit_model(
    final_model,
    train_loader,
    val_loader,
    epochs=30,
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
    focal_alpha=best_params["focal_alpha"],
    focal_gamma=best_params["focal_gamma"],
    patience=7,
)

  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 01/30 | train_loss=0.0257 | val_loss=0.0410 | acc=0.9348 | spoof_rec=0.6162 | spoof_prec=0.4015 | macro_f1=0.7257 | roc_auc=0.8388 | thr=0.31
  *** Improved (spoof_recall=0.6162), saved -> /content/drive/MyDrive/DL Project/saved_models_v7/lr0.000315_decay0.007114/epoch01_spoof_rec0.6162.pt


  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 02/30 | train_loss=0.0124 | val_loss=0.2145 | acc=0.9575 | spoof_rec=0.2650 | spoof_prec=0.6997 | macro_f1=0.6812 | roc_auc=0.8277 | thr=0.02


  0%|          | 0/619 [00:00<?, ?it/s]

...

Epoch 13/30 | train_loss=0.0089 | val_loss=0.0525 | acc=0.9395 | spoof_rec=0.4375 | spoof_prec=0.4032 | macro_f1=0.6939 | roc_auc=0.7772 | thr=0.59


  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 14/30 | train_loss=0.0072 | val_loss=0.0422 | acc=0.9356 | spoof_rec=0.5875 | spoof_prec=0.4017 | macro_f1=0.7214 | roc_auc=0.8172 | thr=0.32


  0%|          | 0/619 [00:00<?, ?it/s]

Epoch 15/30 | train_loss=0.0070 | val_loss=0.0643 | acc=0.9388 | spoof_rec=0.5863 | spoof_prec=0.4203 | macro_f1=0.7285 | roc_auc=0.8054 | thr=0.15
Early stopping: no spoof_recall improvement for 7 epochs.


## 15. Test-set evaluation

In [ ]:
# ─── 15. Test-set evaluation ──────────────────────────────────────────────────
final_model.load_state_dict(final_best_state["model_state_dict"])
best_thr = final_best_state["best_threshold"]

criterion_eval = FocalLoss(
    alpha=best_params["focal_alpha"],
    gamma=best_params["focal_gamma"],
).to(DEVICE)

test_metrics, test_probs, test_labels, _ = evaluate_model(
    final_model, test_loader, criterion_eval, threshold=best_thr
)

print("\n" + "=" * 50)
print("          TEST SET RESULTS")
print("=" * 50)
print(f"  Threshold used      : {best_thr:.3f}")
print(f"  Accuracy            : {test_metrics['accuracy']:.4f}")
print(f"  Balanced accuracy   : {test_metrics['balanced_accuracy']:.4f}")
print(f"  --- Spoof ---")
print(f"  Spoof recall        : {test_metrics['spoof_recall']:.4f}")
print(f"  Spoof precision     : {test_metrics['spoof_precision']:.4f}")
print(f"  Spoof F1            : {test_metrics['spoof_f1']:.4f}")
print(f"  --- Overall ---")
print(f"  Macro F1            : {test_metrics['macro_f1']:.4f}")
print(f"  ROC-AUC             : {test_metrics['roc_auc']:.4f}")
print(f"  EER                 : {test_metrics['eer']:.4f}")
print(f"\nConfusion matrix:")
print(f"               Pred_bonafide  Pred_spoof")
print(f"  Act_bonafide      {test_metrics['tn']:>6}         {test_metrics['fp']:>6}   (FPR={test_metrics['false_positive_rate']:.3f})")
print(f"  Act_spoof         {test_metrics['fn']:>6}         {test_metrics['tp']:>6}   (FNR={test_metrics['false_negative_rate']:.3f})")
print("=" * 50)


          TEST SET RESULTS
  Threshold used      : 0.700
  Accuracy            : 0.5147
  Balanced accuracy   : 0.6331
  --- Spoof ---
  Spoof recall        : 0.7647
  Spoof precision     : 0.0747
  Spoof F1            : 0.1360
  --- Overall ---
  Macro F1            : 0.3993
  ROC-AUC             : 0.7475
  EER                 : 0.3067

Confusion matrix:
               Pred_bonafide  Pred_spoof
  Act_bonafide        7620           7572   (FPR=0.498)
  Act_spoof            188            611   (FNR=0.235)


## 16. Training curves

In [ ]:
# ─── 16. Training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(final_history["train_loss"], label="train")
axes[0].plot(final_history["val_loss"],   label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(final_history["val_spoof_recall"], label="spoof recall", color="red")
axes[1].plot(final_history["val_macro_f1"],     label="macro F1",     color="blue")
axes[1].set_title("Spoof Recall & Macro F1 (val)")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(final_history["val_roc_auc"], label="ROC-AUC", color="green")
axes[2].set_title("ROC-AUC (val)")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
curve_path = str(PROJECT_ROOT / "training_curves_v7.png")
plt.savefig(curve_path, dpi=120)
plt.show()
print(f"Curves saved to {curve_path}")

## 17. Save final model

In [ ]:
# ─── 17. Save final model ─────────────────────────────────────────────────────
final_save_path = SAVE_DIR / "final_model_v7.pt"
torch.save({
    "model_state_dict": final_best_state["model_state_dict"],
    "best_threshold":   best_thr,
    "best_val_metrics": final_best_state["best_val_metrics"],
    "test_metrics":     test_metrics,
    "best_params":      best_params,
}, final_save_path)
print(f"\nFinal model saved to: {final_save_path}")